In [3]:
import os
import cv2
import joblib
import numpy as np
import matplotlib.pyplot as plt

from skimage.feature import local_binary_pattern
from skimage.feature import hog

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier

In [4]:
dataset_path = r"C:\Angel\coin\DataSet"

In [5]:
def extract_features(image_path):

    # Read image
    image = cv2.imread(image_path)

    if image is None:
        return None

    # Resize image
    image = cv2.resize(image, (256, 256))

    # Convert to grayscale
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    # --------------------------------------------------------
    # CIRCULAR HOUGH TRANSFORM
    # --------------------------------------------------------

    circles = cv2.HoughCircles(
        gray,
        cv2.HOUGH_GRADIENT,
        dp=1.2,
        minDist=100,
        param1=100,
        param2=30,
        minRadius=50,
        maxRadius=120
    )

    radius = 0

    if circles is not None:

        circles = np.round(circles[0, :]).astype("int")

        x, y, r = circles[0]

        radius = r

        # Create mask
        mask = np.zeros(gray.shape, dtype="uint8")

        cv2.circle(mask, (x, y), r, 255, -1)

        segmented = cv2.bitwise_and(gray, gray, mask=mask)

    else:

        segmented = gray

    # --------------------------------------------------------
    # FEATURE 1: DIAMETER RATIO
    # --------------------------------------------------------

    diameter_ratio = radius / 128.0

    # --------------------------------------------------------
    # FEATURE 2: EDGE DENSITY
    # --------------------------------------------------------

    edges = cv2.Canny(segmented, 100, 200)

    edge_density = np.sum(edges > 0) / (256 * 256)

    # --------------------------------------------------------
    # FEATURE 3: LBP FEATURES
    # --------------------------------------------------------

    radius_lbp = 3

    points = 8 * radius_lbp

    lbp = local_binary_pattern(
        segmented,
        points,
        radius_lbp,
        method="uniform"
    )

    lbp_hist, _ = np.histogram(
        lbp.ravel(),
        bins=np.arange(0, points + 3),
        range=(0, points + 2)
    )

    lbp_hist = lbp_hist.astype("float")

    lbp_hist /= (lbp_hist.sum() + 1e-6)

    # --------------------------------------------------------
    # FEATURE 4: HOG FEATURES
    # --------------------------------------------------------

    hog_features = hog(
        segmented,
        orientations=9,
        pixels_per_cell=(16, 16),
        cells_per_block=(2, 2),
        block_norm='L2-Hys',
        feature_vector=True
    )

    hog_features = hog_features[:200]

    # --------------------------------------------------------
    # FEATURE 5: HSV HISTOGRAM
    # --------------------------------------------------------

    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)

    hist = cv2.calcHist(
        [hsv],
        [0, 1],
        None,
        [8, 8],
        [0, 180, 0, 256]
    )

    hist = cv2.normalize(hist, hist).flatten()

    # --------------------------------------------------------
    # COMBINE FEATURES
    # --------------------------------------------------------

    features = np.hstack([
        diameter_ratio,
        edge_density,
        lbp_hist,
        hog_features,
        hist
    ])

    return features

In [6]:
# ============================================================
# LOAD DATASET
# ============================================================

X = []
y = []

valid_extensions = ('.jpg', '.jpeg', '.png')

for root, dirs, files in os.walk(dataset_path):

    for file in files:

        if not file.lower().endswith(valid_extensions):
            continue

        path = os.path.join(root, file)

        folder_names = root.split(os.sep)

        label = None

        for folder in folder_names:

            folder_lower = folder.lower()

            if 'one' in folder_lower:
                label = '1'

            elif 'two' in folder_lower:
                label = '2'

            elif 'five' in folder_lower:
                label = '5'

            elif 'ten' in folder_lower:
                label = '10'

            elif 'twenty' in folder_lower:
                label = '20'

        if label is None:
            continue

        features = extract_features(path)

        if features is not None:

            X.append(features)

            y.append(label)

X = np.array(X)

y = np.array(y)

print("Dataset Loaded Successfully")
print("Total Samples:", len(X))

Dataset Loaded Successfully
Total Samples: 900
